# 标签噪声与错误分析

**负责人：** 2 号　**阶段：** 4 分析

解释模型在哪里出错，以及其中有多少是标签本身的问题。

## 任务

标"（选做）"的任务在必做部分完成后再做。

- [x] 计算 VADER 和给定标签的一致率，检验"标签由 VADER 生成"这个假设（只做分析，绝不能当特征）
- [x] 错误分析代码（Part B）：找出所有模型都分错的推文，看它们和 VADER 不一致、金融用语的关系，导出表 8 的候选例子
- [x] 训练时去掉 VADER 不一致的推文：kNN 已在 `21_m2_knn` 里做过，CV 分数下降
- [ ] 11/15 全员预测到齐后重跑 Part B，读候选例子并填写 `likely_cause`，结果交给 3 号和 6 号

## 产出

- 报告 4.3 节的表 8 和错误规律
- `figures/analysis_*.png`
- `DATA_ROOT/clean_v2/error_cases_v2.csv`（表 8 候选例子，不进 Git）

In [ ]:
import sys
from pathlib import Path

# 向上查找仓库根目录（含 src/config.py 的文件夹），在任何子文件夹里打开都能导入 src
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("找不到仓库根目录：请在 clone 下来的仓库里启动 Jupyter（Colab 先 %cd 进仓库）")
sys.path.insert(0, str(ROOT))

from src.config import DATA_ROOT, FIGURES_DIR, LABELS, SEED                 # 路径和常量
from src.data import load_clean, load_raw, load_splits                      # 读取数据和划分
from src.evaluate import evaluate, new_run_id, plot_confusion, save_result  # 统一评分
from src.cache import load_embedding, save_embedding, save_preds            # 嵌入和预测结果的读写

print("仓库根目录:", ROOT)
print("DATA_ROOT:", DATA_ROOT)

# Part A: where do the labels come from?

The dataset description says the tweets were labelled Positive / Neutral / Negative, but not **how**. If the labels were produced by a sentiment lexicon rather than by people, three things follow:

1. our classifiers learn to imitate that tool, so its blind spots become our "ground truth";
2. the tool's score must **never** be used as a feature (it would leak the label);
3. disagreements between the labels and real financial meaning are a business insight, not just noise.

**Hypothesis:** the labels are VADER compound scores thresholded at ±0.05 (the threshold recommended by VADER's authors), computed on the original tweet text.

**Test:** if the hypothesis is true, VADER should reproduce the labels far better than an unrelated sentiment lexicon (TextBlob), work best on the raw text, and agree *more* on tweets VADER cannot read (non-English → score 0 → Neutral).

The test uses every readable, non-duplicate tweet from `clean_v2_all.csv`: the retained tweets plus those removed as non-English or word-less, because those removed groups are part of the evidence.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from src.config import CLEAN_DIR, CLEAN_VERSION

full = pd.read_csv(CLEAN_DIR / f"clean_v{CLEAN_VERSION}_all.csv", keep_default_na=False)
readable = full["keep"].eq(1) | full["drop_reason"].isin(["non_english", "non_english_manual", "no_content"])
df = full[readable].reset_index(drop=True)
sia = SentimentIntensityAnalyzer()


def to_label(score, threshold):
    """Map a polarity score to the dataset's codes: 1 Positive, 2 Negative, 0 Neutral."""
    return np.where(score >= threshold, 1, np.where(score <= -threshold, 2, 0))


df["vader_raw"] = df["text_raw"].map(lambda s: sia.polarity_scores(s)["compound"])
df["vader_clean"] = df["text_clean"].map(lambda s: sia.polarity_scores(s)["compound"])
df["textblob_raw"] = df["text_raw"].map(lambda s: TextBlob(s).sentiment.polarity)
print(f"{len(df)} readable tweets (clean_v{CLEAN_VERSION}): {int(df['keep'].sum())} retained + "
      f"{int((df['keep'] == 0).sum())} removed as non-English or word-less")

## A1. Does VADER reproduce the labels better than another sentiment tool?

In [ ]:
candidates = {
    "VADER, raw text, ±0.05": to_label(df["vader_raw"], 0.05),
    "VADER, raw text, any non-zero": to_label(df["vader_raw"], 1e-9),
    "VADER, cleaned text_clean, ±0.05": to_label(df["vader_clean"], 0.05),
    "TextBlob, raw text, any non-zero": to_label(df["textblob_raw"], 1e-9),
    "Majority class (always Positive)": np.ones(len(df), dtype=int),
}
agreement = pd.DataFrame([
    {"labeller": name, "agreement": (pred == df["label"]).mean(), "Cohen's kappa": cohen_kappa_score(df["label"], pred)}
    for name, pred in candidates.items()
]).round(3)
agreement

VADER on the raw text reproduces the given labels far better than TextBlob, a different lexicon, and the agreement is "almost perfect" by Cohen's kappa. Agreement **drops** on the cleaned text, mostly because restored contractions add negations the labeller never saw (see `data/clean_v2/clean_v2_log.md`).

## A2. Where does VADER disagree with the labels?

In [ ]:
df["vader_label"] = candidates["VADER, raw text, ±0.05"]
df["agree"] = df["vader_label"] == df["label"]


def group(row):
    if row["drop_reason"].startswith("non_english"):
        return "Non-English (removed)"
    if row["drop_reason"] == "no_content":
        return "No words (removed)"
    return "Too short to detect" if row["lang"] == "short" else "English"


by_lang = df.groupby(df.apply(group, axis=1))["agree"].agg(agreement="mean", tweets="size").round(3)
by_lost = df.groupby(np.where(df["emoji_lost"] == 1, "has lost characters", "no lost characters"))["agree"].agg(
    agreement="mean", tweets="size").round(3)
pd.concat({"language": by_lang, "characters lost in the source file": by_lost})

- **Non-English and word-less tweets agree most.** VADER only knows English words, so it scores them 0 and they get *Neutral*, which is what the labels say for nearly all of them. This is why `clean_v2` removes them: their labels say nothing about sentiment.
- **Tweets with lost characters agree less.** Emoji carry sentiment in VADER. If the labels were computed before the emoji were destroyed in the file we received, VADER on today's text cannot see them, so these tweets disagree more often.

In [ ]:
fig = plot_confusion(df["label"], df["vader_label"], "Given label vs VADER (raw text, ±0.05)",
                     save_as=FIGURES_DIR / "analysis_vader_vs_labels.png")

In [ ]:
# Disagreements among the retained tweets are candidates for label noise; 21_m2_knn compares them with its own flags
retained = df["keep"] == 1
disagree = df.loc[retained & ~df["agree"], ["id", "label", "vader_label", "vader_raw", "lang", "emoji_lost", "text_raw"]].copy()
disagree["label"] = disagree["label"].map(LABELS)
disagree["vader_label"] = disagree["vader_label"].map(LABELS)
disagree.to_csv(CLEAN_DIR / f"vader_disagreements_v{CLEAN_VERSION}.csv", index=False, encoding="utf-8")
print(f"{len(disagree)} disagreements among {int(retained.sum())} retained tweets saved")
disagree.sample(10, random_state=SEED)

## A3. What this means for the project

- **The hypothesis is strongly supported**, but it remains a hypothesis: we cannot see the original pipeline, and the remaining disagreements can come from emoji lost in the file, restored contractions, a different VADER version or preprocessing we cannot reproduce.
- **The task is effectively "imitate VADER".** Models that rely on sentiment-bearing words have an advantage; semantic models such as SBERT may "disagree" with the labels when they understand finance better than VADER, which gives no score at all to *bearish*, *bullish* or *short*.
- **VADER scores must never be a model input.** They are used here only to analyse the labels.
- **Business insight to develop in Part B:** where the models and VADER both misread financial language, a generic sentiment tool would give a misleading signal to an investor dashboard.

# Part B: which tweets do the models get wrong?

Part B uses the **out-of-fold** predictions of each model's best cross-validation run on the training split, so the test split stays untouched. It works with whatever models have saved predictions so far. **Rerun it after 15 Nov**, when the baseline and all six models are in; until then the numbers below describe only the models listed in B1.

Removing the A2 disagreements from the training folds was tested for kNN in `21_m2_knn`: the CV macro-F1 went down (0.456 against 0.473 with FastText vectors), so all tweets stay in training.

## B1. Collect the predictions

In [ ]:
from scipy.stats import fisher_exact

from src.cache import list_preds, load_preds
from src.evaluate import collect_results

clean = load_clean()
splits = load_splits(clean=clean)
dev = splits.loc[splits["split"] == "train", ["id"]].merge(clean, on="id").reset_index(drop=True)

# Best CV run of each model among the runs that saved predictions
results = collect_results()
results = results[(results["split"] == "cv") & results["run_id"].isin(list_preds())]
runs = results.loc[results.groupby("model")["macro_f1"].idxmax()].sort_values("macro_f1", ascending=False)

predicted = pd.DataFrame(index=dev.index)
for _, run in runs.iterrows():
    saved = load_preds(run["run_id"])
    oof = pd.Series(np.asarray(saved["oof_proba"]).argmax(1), index=np.asarray(saved["oof_ids"]))
    if set(oof.index) != set(dev["id"]):
        raise ValueError(f"{run['run_id']} does not cover the training split; check its split version")
    predicted[run["model"]] = dev["id"].map(oof).to_numpy()
models = list(predicted.columns)
print(f"{len(models)} model(s) with saved predictions: {models}")
if len(models) < 7:
    print("Not all models are in yet (baseline + 6 expected); rerun Part B after 15 Nov.")
runs[["model", "embedding", "run_id", "macro_f1"]].round(3)

## B2. Tweets that every model gets wrong

If a tweet is misclassified by every model, the cause is more likely the tweet or its label than one model's weakness. We compare these tweets with the rest on two signals from Part A: whether VADER disagrees with the given label, and whether the source file destroyed characters (`emoji_lost`).

In [ ]:
wrong = predicted.ne(dev["label"], axis=0)
dev["n_models_wrong"] = wrong.sum(axis=1)
dev["all_wrong"] = wrong.all(axis=1)
vader_ids = set(pd.read_csv(CLEAN_DIR / f"vader_disagreements_v{CLEAN_VERSION}.csv")["id"])
dev["vader_disagrees"] = dev["id"].isin(vader_ids)

group_names = dev["all_wrong"].map({True: "wrong in every model", False: "right in at least one model"})
summary = dev.groupby(group_names).agg(
    tweets=("id", "size"),
    vader_disagreement_rate=("vader_disagrees", "mean"),
    emoji_lost_rate=("emoji_lost", "mean"),
).round(3)
print(f"{int(dev['all_wrong'].sum())} of {len(dev)} training tweets are wrong in all {len(models)} model(s)")
summary

In [ ]:
# Given labels of the tweets every model gets wrong, compared with all training tweets
pd.DataFrame({
    "wrong in every model": dev.loc[dev["all_wrong"], "label_name"].value_counts(normalize=True),
    "all training tweets": dev["label_name"].value_counts(normalize=True),
}).round(3)

## B3. Are finance terms over-represented in these errors?

VADER gives no score to market words such as *bearish*, *bullish*, *short* or *dip*, and it reads *crash* as negative even in a tweet that sees a buying chance. If the models learned VADER's view, tweets with these words should be wrong more often. We count the terms in `text_clean` as whole words, so hashtags such as *stockmarketcrash* do not count, and use Fisher's exact test on the 2×2 table (term present or not, wrong in every model or not). An odds ratio above 1 means the terms are more common among the errors; a p-value below 0.05 means the difference is unlikely to be chance. *Puts* can also be an ordinary verb, so check its examples by hand.

In [ ]:
FINANCE_TERMS = ["bear", "bearish", "bull", "bullish", "crash", "dip", "short", "puts", "red", "green"]
words = dev["text_clean"].str.lower()
has_term = pd.DataFrame({term: words.str.contains(rf"\b{term}\b") for term in FINANCE_TERMS})
dev["finance_term"] = has_term.any(axis=1)

table = pd.crosstab(dev["all_wrong"], dev["finance_term"]).reindex(index=[False, True], columns=[False, True], fill_value=0)
odds_ratio, p_value = fisher_exact(table)
rate = dev.groupby("all_wrong")["finance_term"].mean()
print(f"tweets with a finance term: {rate.get(True, 0):.1%} of those wrong in every model, "
      f"{rate.get(False, 0):.1%} of the others; odds ratio {odds_ratio:.2f}, p = {p_value:.3f}")

per_term = pd.DataFrame({
    "tweets": has_term.sum(),
    "wrong in every model": has_term[dev["all_wrong"]].sum(),
})
per_term["share wrong"] = (per_term["wrong in every model"] / per_term["tweets"]).round(3)
per_term.sort_values("tweets", ascending=False)

## B4. Candidate examples for Table 8

The report needs a few representative errors with their likely cause (Table 8 of the template). The file below lists every tweet that all models get wrong, with the most common wrong prediction and the three signals above. Read the tweets and fill in `likely_cause` by hand, for example *label follows VADER, text is positive*, *finance term misread*, *sarcasm*, *emoji lost*.

In [ ]:
wrong_rows = predicted.loc[dev["all_wrong"]]
most_common_wrong = wrong_rows.mode(axis=1)[0].astype(int).map(LABELS) if len(wrong_rows) else pd.Series(dtype=object)
cases = dev.loc[dev["all_wrong"], ["id", "label_name", "vader_disagrees", "finance_term", "emoji_lost", "text_raw"]].copy()
cases.insert(2, "predicted", most_common_wrong)
cases["likely_cause"] = ""
out = CLEAN_DIR / f"error_cases_v{CLEAN_VERSION}.csv"
cases.to_csv(out, index=False, encoding="utf-8")
print(f"{len(cases)} cases saved to {out.name}")
cases.sample(min(10, len(cases)), random_state=SEED)